# Data Preparation: Synthetic Healthcare Patient Visit Data

## Purpose

This notebook performs initial data preparation for a synthetic healthcare dataset.
The goal is to produce a clean, validated patient-visit–level dataset suitable for downstream
exploratory analysis and modeling.


## Data Loading & Schema Inspection


In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from pandas import json_normalize


DATA_DIR = Path("../data/raw")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
patients = pd.read_csv(DATA_DIR / "patients.csv")
visits_q1 = pd.read_csv(DATA_DIR / "visits_q1.csv")
visits_q2 = pd.read_csv(DATA_DIR / "visits_q2.csv")
region_lookup = pd.read_excel(DATA_DIR / "region_lookup copy.xlsx")

import json
with open(DATA_DIR / "procedure_catalog.json", "r") as f:
    procedure_catalog = json.load(f)

In [3]:
datasets = {
    "patients": patients,
    "visits_q1": visits_q1,
    "visits_q2": visits_q2,
    "region_lookup": region_lookup,
}

# Convert procedure_catalog (nested lists) to DataFrame
procedure_catalog_df = json_normalize(procedure_catalog)
datasets["procedure_catalog"] = procedure_catalog_df


for name, df in datasets.items():
  print(f"\n{'='*60}")
  print(f"Structure of {name}")
  print(f"{'='*60}")

  print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
  print("\nColumn names and data types:")
  print(df.dtypes)

  print("\nNon-null count and memory info:")
  df.info()


Structure of patients
Shape: 123 rows x 5 columns

Column names and data types:
Patient_ID     object
name           object
email          object
birth_date     object
region_code    object
dtype: object

Non-null count and memory info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Patient_ID   123 non-null    object
 1   name         123 non-null    object
 2   email        123 non-null    object
 3   birth_date   123 non-null    object
 4   region_code  100 non-null    object
dtypes: object(5)
memory usage: 4.9+ KB

Structure of visits_q1
Shape: 151 rows x 7 columns

Column names and data types:
Visit_ID           object
Patient_ID         object
Clinic_ID          object
Visit_Date         object
Procedure_Code     object
Amount            float64
Quantity          float64
dtype: object

Non-null count and memory info:
<class 'pandas.core.

In [4]:
for name, df in datasets.items():
  print(f"\n{'='*60}")
  print(f"Sample rows from {name}")
  print(f"{'='*60}")
  print(df.head(5))
  print(df.tail(5))


Sample rows from patients
  Patient_ID            name                       email  birth_date  \
0     PT1000     Casey Smith        casey.smith@mail.com  1985-11-08   
1     PT1001     Riley Brown     riley.brown@example.com  1994-05-07   
2     PT1002    Jordan Smith       jordan.smith@mail.com  1957-03-22   
3     PT1003    Dakota Smith     dakota.smith@clinic.net  1986-02-24   
4     PT1004  Sawyer Jackson  sawyer.jackson@example.com  1963-06-02   

  region_code  
0          SE  
1         NaN  
2          Sw  
3          SE  
4          SW  
    Patient_ID            name                      email  birth_date  \
118     PT1118    Parker Brown    parker.brown@health.org  1965-06-23   
119     PT1119   Hayden Miller     hayden.miller@mail.com  1999-11-10   
120     PTDUP1    Finley Brown   finley.brown@example.com  1947-07-27   
121     PTDUP2  Reese Williams  reese.williams@health.org  1966-08-07   
122     PTDUP3  Emerson Martin  emerson.martin@health.org  1981-08-14   

    r

In [5]:
# Function to detect and report duplicate rows in each dataset
def check_duplicates(datasets):
  for name, df in datasets.items():
    print(f"\n{name} - Duplicate Rows Check:")
    total_rows = len(df)
    dup_rows = df.duplicated()
    dup_count = dup_rows.sum()
    dup_percent = round((dup_count / total_rows) * 100, 2)

    if dup_count > 0:
      print(f"Found {dup_count} duplicate rows ({dup_percent}%)")
      print("Sample duplicates:\n", df[dup_rows].head())
    else:
      print("No duplicate rows found.")

check_duplicates(datasets)


patients - Duplicate Rows Check:
No duplicate rows found.

visits_q1 - Duplicate Rows Check:
Found 1 duplicate rows (0.66%)
Sample duplicates:
       Visit_ID Patient_ID Clinic_ID  Visit_Date Procedure_Code  Amount  \
150  VQ1-10041     PT1084     CL102  2024-02-12        PROC007   101.2   

     Quantity  
150       1.0  

visits_q2 - Duplicate Rows Check:
No duplicate rows found.

region_lookup - Duplicate Rows Check:
No duplicate rows found.

procedure_catalog - Duplicate Rows Check:
No duplicate rows found.


In [6]:
# Function to display missing value summary
def check_missing_values(datasets):
  for name, df in datasets.items():
    print(f"\n{name} - Missing Values Summary")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
      print(missing.sort_values(ascending=False))
    else:
      print("No missing values found.")

check_missing_values(datasets)


patients - Missing Values Summary
region_code    23
dtype: int64

visits_q1 - Missing Values Summary
No missing values found.

visits_q2 - Missing Values Summary
clinicId         63
patientId         1
visitDate         1
procedureCode     1
quantity          1
amount            1
dtype: int64

region_lookup - Missing Values Summary
No missing values found.

procedure_catalog - Missing Values Summary
Standard_Cost    1
dtype: int64


**Issues Noted**:

  - patients.csv: region code 23 null values \
  inconsistent region code casing: "SW" vs "Sw" (row 2 vs row 4) \
  birth_date should be datetime type\
  Patient_ID in Pascal Case
  - visits_q1.csv: 1 duplicated row. \
  wrong column case \
  visit_date should be datetime type.  \
  row 1 has Amount = -479.50 (possibly an error, or an outlier refund)
  - visits_q2.csv: amount should be changed to a numeric value, clinicId column has 63 null values; \  
  wrong column case \
  the other columns all have 1 missing value. (total is missing) \
  visit_date should be datetime type \
  visits_q1 uses YYYY-MM-DD while visits_q2 uses MM/DD/YYYY \
  visits_q1 & visits_q2 have different naming conventions
  - region_lookup.xlsx: Region_Code and Region_Name in Pascal Case, maybe consider switch to lowercase.
  - procedure_catalog.json: Standard_Cost column has 1 null value\
  wrong column case



---



## Data Cleaning & Type Correction

Clean each dataset so it is analysis-ready:
- Fix types
- Handle invalid values
- Standardize naming
- Remove or flag problematic rows


In [7]:
# Standardize all column names
def standardize_column_names(df):
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df

# Apply to all dataframes
visits_q1 = standardize_column_names(visits_q1)
visits_q2 = standardize_column_names(visits_q2)
patients = standardize_column_names(patients)
# Apply to the DataFrame, not the original list
procedure_catalog_df = standardize_column_names(procedure_catalog_df)
region_lookup = standardize_column_names(region_lookup)

In [8]:
# Change amount to numeric type
visits_q2['amount'] = visits_q2['amount'].astype(str).str.replace('$', '',
                                                                  regex=False)
visits_q2['amount'] = pd.to_numeric(visits_q2['amount'], errors='coerce')

# Validate: check for remaining non-numeric values
print(visits_q2.dtypes)

visitid           object
patientid         object
clinicid          object
visitdate         object
procedurecode     object
quantity         float64
amount           float64
dtype: object


In [9]:
# Define the desired column order for visits_q2 to match the initial visits_q1 structure
visits_q2_desired_order = ['visitid', 'patientid', 'clinicid',
                           'visitdate', 'procedurecode', 'quantity', 'amount']

# Reindex visits_q2 to match the desired order
visits_q2 = visits_q2[visits_q2_desired_order]

visits_q2.columns = visits_q1.columns
visits_q2.head()

,visit_id,patient_id,clinic_id,visit_date,procedure_code,amount,quantity
0,VQ2-20000,PT1095,NaN,06/24/2024,PROC005,1.0,282.74
1,VQ2-20001,PT1057,NaN,04/07/2024,PROC021,2.0,189.71
2,VQ2-20002,PT1060,NaN,05/18/2024,PROC025,3.0,225.22
3,VQ2-20003,PT1031,CL107,04/07/2024,PROC016,3.0,156.47
4,VQ2-20004,PT1034,CL104,04/07/2024,PROC027,3.0,274.67


In [10]:
# Check and remove duplicates in visits_q1
print("Visits Q1 duplicates:", visits_q1.duplicated().sum())
visits_q1 = visits_q1.drop_duplicates()

# Remove the "TOTAL" row from visits_q2 if it exists
visits_q2 = visits_q2[visits_q2['visit_id'] != 'TOTAL']
visits_q2 = visits_q2.drop_duplicates()

# Check for duplicates in other datasets
print("Visits Q2 duplicates:", visits_q2.duplicated().sum())
print("Patients duplicates:", patients.duplicated().sum())

Visits Q1 duplicates: 1
Visits Q2 duplicates: 0
Patients duplicates: 0


In [11]:
# Convert 'birth_date' in patients to datetime, handling errors by coercing invalid parsing to NaT
patients['birth_date'] = pd.to_datetime(patients['birth_date'], errors='coerce')

# Display the dtypes to confirm the change
print("patients dtypes after birth_date conversion:")
print(patients.dtypes)

# Display the first few rows to show the change
print("\npatients head after birth_date conversion:")
display(patients.head())

# Check how many values were coerced to NaT
print("\nNumber of invalid birth_date values (coerced to NaT):")
print(patients['birth_date'].isna().sum())

patients dtypes after birth_date conversion:
patient_id             object
name                   object
email                  object
birth_date     datetime64[ns]
region_code            object
dtype: object

patients head after birth_date conversion:


,patient_id,name,email,birth_date,region_code
0,PT1000,Casey Smith,casey.smith@mail.com,1985-11-08,SE
1,PT1001,Riley Brown,riley.brown@example.com,1994-05-07,NaN
2,PT1002,Jordan Smith,jordan.smith@mail.com,1957-03-22,Sw
3,PT1003,Dakota Smith,dakota.smith@clinic.net,1986-02-24,SE
4,PT1004,Sawyer Jackson,sawyer.jackson@example.com,1963-06-02,SW



Number of invalid birth_date values (coerced to NaT):
3


In [12]:
# Convert 'Visit_Date' to datetime objects in both dataframes
visits_q1['visit_date'] = pd.to_datetime(visits_q1['visit_date'])
visits_q2['visit_date'] = pd.to_datetime(visits_q2['visit_date'])


# Display the dtypes to confirm the change
print("visits_q1 dtypes after date conversion and formatting:")
print(visits_q1.dtypes)
print("\nvisits_q2 dtypes after date conversion:")
print(visits_q2.dtypes)

# Display the first few rows to show the format change
print("\nvisits_q1 head after date formatting:")
display(visits_q1.head())
print("\nvisits_q2 head after date conversion:")
display(visits_q2.head())

visits_q1 dtypes after date conversion and formatting:
visit_id                  object
patient_id                object
clinic_id                 object
visit_date        datetime64[ns]
procedure_code            object
amount                   float64
quantity                 float64
dtype: object

visits_q2 dtypes after date conversion:
visit_id                  object
patient_id                object
clinic_id                 object
visit_date        datetime64[ns]
procedure_code            object
amount                   float64
quantity                 float64
dtype: object

visits_q1 head after date formatting:


/var/folders/08/9mv7pyp94lxd285_5f6x1n5m0000gn/T/ipykernel_59234/952796271.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  visits_q1['visit_date'] = pd.to_datetime(visits_q1['visit_date'])


,visit_id,patient_id,clinic_id,visit_date,procedure_code,amount,quantity
0,VQ1-10000,PT1026,CL103,2024-01-17,PROC013,96.02,1.0
1,VQ1-10001,PT1087,CL104,2024-01-16,PROC026,-479.50,1.0
2,VQ1-10002,PT1015,CL109,2024-01-06,PROC023,61.93,2.0
3,VQ1-10003,PT1010,CL101,2024-03-10,PROC014,32.60,1.0
4,VQ1-10004,PT1087,CL108,2024-03-06,PROC017,187.82,1.0



visits_q2 head after date conversion:


,visit_id,patient_id,clinic_id,visit_date,procedure_code,amount,quantity
0,VQ2-20000,PT1095,NaN,2024-06-24,PROC005,1.0,282.74
1,VQ2-20001,PT1057,NaN,2024-04-07,PROC021,2.0,189.71
2,VQ2-20002,PT1060,NaN,2024-05-18,PROC025,3.0,225.22
3,VQ2-20003,PT1031,CL107,2024-04-07,PROC016,3.0,156.47
4,VQ2-20004,PT1034,CL104,2024-04-07,PROC027,3.0,274.67


In [13]:
# Check for negative amounts and outliers
print("Negative amounts in visits:")
print("Q1:", (visits_q1['amount'] < 0).sum())
print("Q2:", (visits_q2['amount'] < 0).sum())

# Check quantity values
print("Invalid quantities (<=0):")
print("Q1:", (visits_q1['quantity'] <= 0).sum())
print("Q2:", (visits_q2['quantity'] <= 0).sum())

# Check for duplicate region codes in region_lookup
print("Duplicate region codes:",
      region_lookup.duplicated(subset=['region_code']).sum())
region_lookup = region_lookup.drop_duplicates(subset=['region_code'])

Negative amounts in visits:
Q1: 10
Q2: 0
Invalid quantities (<=0):
Q1: 0
Q2: 7
Duplicate region codes: 1


In [14]:
# Check if all patients in visits exist in patients master
missing_patients_q1 = set(visits_q1['patient_id']) - set(patients['patient_id'])
missing_patients_q2 = set(visits_q2['patient_id']) - set(patients['patient_id'])
print("Patients in visits but not in patient master:")
print("Q1:", len(missing_patients_q1))
print("Q2:", len(missing_patients_q2))

# Check if all procedures exist in catalog
missing_proc_q1 = set(visits_q1['procedure_code']) - set(procedure_catalog_df['procedure_code'])
missing_proc_q2 = set(visits_q2['procedure_code']) - set(procedure_catalog_df['procedure_code'])
print("Procedures in visits but not in catalog:")
print("Q1:", len(missing_proc_q1))
print("Q2:", len(missing_proc_q2))

Patients in visits but not in patient master:
Q1: 0
Q2: 0
Procedures in visits but not in catalog:
Q1: 0
Q2: 0


In [15]:
# Fix Standard_Cost data type and handle null value
procedure_catalog_df['standard_cost'] = \
pd.to_numeric(procedure_catalog_df['standard_cost'], errors='coerce')
print("Null Standard_Cost:", procedure_catalog_df['standard_cost'].isna().sum())

# Standardize Specialty casing
procedure_catalog_df['specialty'] = \
procedure_catalog_df['specialty'].str.title()

Null Standard_Cost: 1


In [16]:
# Standardize the region_lookup codes to uppercase for consistency
region_lookup['region_code'] = region_lookup['region_code'].str.upper()

# Standardize patients region codes to uppercase
patients['region_code'] = patients['region_code'].str.strip().str.upper()

print('Unique region codes:', patients['region_code'].dropna().unique())


# Check results
print("\nAfter standardization:")
print("Unique region codes in patients:", patients['region_code'].unique())
print("Missing region codes:", patients['region_code'].isna().sum())

# Verify against region_lookup
valid_regions = set(region_lookup['region_code'])
patient_regions = set(patients['region_code'].dropna())
print("All patient regions are valid:", patient_regions.issubset(valid_regions))

Unique region codes: ['SE' 'SW' 'WC' 'NE' 'MW' 'NW']

After standardization:
Unique region codes in patients: ['SE' nan 'SW' 'WC' 'NE' 'MW' 'NW']
Missing region codes: 23
All patient regions are valid: True


/var/folders/08/9mv7pyp94lxd285_5f6x1n5m0000gn/T/ipykernel_59234/3486007555.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  region_lookup['region_code'] = region_lookup['region_code'].str.upper()


In [17]:
# Check for duplicates in region_lookup (I see "NW" appears twice)
print("Region lookup duplicates:")
print(region_lookup[region_lookup.duplicated(subset=['region_code'], keep=False)])

# Remove duplicates, keeping the first occurrence
region_lookup = region_lookup.drop_duplicates(subset=['region_code'], keep='first')

Region lookup duplicates:
Empty DataFrame
Columns: [region_code, region_name]
Index: []


In [18]:
# Create a summary of the current data quality

def data_quality_summary():
    print("=== FINAL DATA QUALITY SUMMARY ===")
    print(f"Visits Q1: {len(visits_q1)} records")
    print(f"Visits Q2: {len(visits_q2)} records")
    print(f"Patients: {len(patients)} records")
    print(f"Procedures: {len(procedure_catalog)} records")
    print(f"Regions: {len(region_lookup)} records")

    print("\nData types confirmed:")
    print("Visit dates:", visits_q1['visit_date'].dtype, visits_q2['visit_date'].dtype)
    print("Amounts numeric:", visits_q1['amount'].dtype, visits_q2['amount'].dtype)

data_quality_summary()

=== FINAL DATA QUALITY SUMMARY ===
Visits Q1: 150 records
Visits Q2: 150 records
Patients: 123 records
Procedures: 30 records
Regions: 6 records

Data types confirmed:
Visit dates: datetime64[ns] datetime64[ns]
Amounts numeric: float64 float64




---



## Dataset Integration

1. Combine visits from Q1 and Q2 into a single table.
2. Join visits with patients, procedures, and regions.
3. Use left joins so no visits are dropped.
4. Display the first few rows of merged and integrated dataframe

In [19]:
# Step 1: Vertical merge of visits
visits_q1['source'] = 'Q1'
visits_q2['source'] = 'Q2'

visits = pd.concat([visits_q1, visits_q2], ignore_index=True)

# Step 2: Join with patient data
visits = pd.merge(visits, patients, how='left', on='patient_id')

# Step 3: Join with procedure catalog
visits = pd.merge(visits, procedure_catalog_df, how='left', on='procedure_code')

# Step 4: Join with region lookup
visits = pd.merge(visits, region_lookup, how='left', on='region_code')

display(visits.head())
print(visits.shape)

/var/folders/08/9mv7pyp94lxd285_5f6x1n5m0000gn/T/ipykernel_59234/788979437.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  visits_q1['source'] = 'Q1'


,visit_id,patient_id,clinic_id,visit_date,procedure_code,amount,quantity,source,name,email,birth_date,region_code,procedure_name,specialty,standard_cost,region_name
0,VQ1-10000,PT1026,CL103,2024-01-17,PROC013,96.02,1.0,Q1,Emerson Martin,emerson.martin@health.org,1981-08-14,SW,Imaging 23,Radiology,822.83,Southwest
1,VQ1-10001,PT1087,CL104,2024-01-16,PROC026,-479.50,1.0,Q1,Sawyer Miller,sawyer.miller@mail.com,1970-11-23,NE,Procedure 42,Neurology,421.40,Northeast
2,VQ1-10002,PT1015,CL109,2024-01-06,PROC023,61.93,2.0,Q1,Jordan Davis,jordan.davis@health.org,1993-02-26,NaN,Lab Panel 30,Urology,671.73,NaN
3,VQ1-10003,PT1010,CL101,2024-03-10,PROC014,32.60,1.0,Q1,Parker Williams,parker.williams@example.com,1948-03-12,NaN,Therapy 25,Oncology,228.30,NaN
4,VQ1-10004,PT1087,CL108,2024-03-06,PROC017,187.82,1.0,Q1,Sawyer Miller,sawyer.miller@mail.com,1970-11-23,NE,Screening 42,Dermatology,995.99,Northeast


(300, 16)




---



## Validation Checks

Check final merged dataset in answering the following questions:
- Are there missing patients, procedures, or regions?
- Are there duplicate Visit_IDs?
- Are there implausible values (negative or extreme amounts, unrealistic birthdates)?
- Summarize findings.


In [20]:
# Show number of missing values per column
missing_counts = visits.isnull().sum()
print("Missing values per column:\n", missing_counts)

Missing values per column:
 visit_id           0
patient_id         0
clinic_id         62
visit_date         0
procedure_code     0
amount             0
quantity           1
source             0
name               0
email              0
birth_date         7
region_code       69
procedure_name     0
specialty          0
standard_cost     10
region_name       69
dtype: int64


In [21]:
# Check for missing values in key fields
key_fields = ['clinic_id', 'region_code']
missing_key_rows = visits[visits[key_fields].isnull().any(axis=1)]

print(f"Rows with missing key fields: {len(missing_key_rows)}")
if not missing_key_rows.empty:
  display(missing_key_rows) # Show all rows with missing keys

Rows with missing key fields: 121


,visit_id,patient_id,clinic_id,visit_date,procedure_code,amount,quantity,source,name,email,birth_date,region_code,procedure_name,specialty,standard_cost,region_name
2,VQ1-10002,PT1015,CL109,2024-01-06,PROC023,61.93,2.00,Q1,Jordan Davis,jordan.davis@health.org,1993-02-26,NaN,Lab Panel 30,Urology,671.73,NaN
3,VQ1-10003,PT1010,CL101,2024-03-10,PROC014,32.60,1.00,Q1,Parker Williams,parker.williams@example.com,1948-03-12,NaN,Therapy 25,Oncology,228.30,NaN
6,VQ1-10006,PT1078,CL103,2024-01-09,PROC021,216.12,1.00,Q1,Casey Martin,casey.martin@health.org,1969-03-26,NaN,Imaging 39,Urology,148.15,NaN
19,VQ1-10019,PT1106,CL103,2024-02-20,PROC026,195.04,1.00,Q1,Avery Rodriguez,avery.rodriguez@example.com,1994-03-04,NaN,Procedure 42,Neurology,421.40,NaN
21,VQ1-10021,PT1042,CL106,2024-01-09,PROC026,123.13,2.00,Q1,Morgan Miller,morgan.miller@mail.com,1980-03-01,NaN,Procedure 42,Neurology,421.40,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,VQ2-20142,PT1039,NaN,2024-05-22,PROC010,2.00,229.05,Q2,Reese Miller,reese.miller@clinic.net,1956-08-05,WC,Lab Panel 16,Pediatrics,108.38,West Coast
294,VQ2-20144,PT1046,CL104,2024-06-07,PROC018,1.00,347.71,Q2,Hayden Johnson,hayden.johnson@example.com,1984-07-17,NaN,Follow-up 43,Pediatrics,196.12,NaN
295,VQ2-20145,PT1031,NaN,2024-04-01,PROC016,2.00,34.89,Q2,Rowan Moore,rowan.moore@clinic.net,2004-09-24,SE,Follow-up 22,Neurology,337.79,Southeast
296,VQ2-20146,PT1113,NaN,2024-05-22,PROC017,2.00,129.75,Q2,Finley Johnson,finley.johnson@health.org,1996-05-23,SE,Screening 42,Dermatology,995.99,Southeast


In [22]:
# Check how many duplicate visit_ids exist
duplicate_key_mask = visits.duplicated(subset='visit_id', keep=False)
num_duplicate_keys = duplicate_key_mask.sum()

print(f"Number of duplicate visit_ids: {num_duplicate_keys}")

Number of duplicate visit_ids: 0


In [23]:
# Check how many full row duplicates exist
full_duplicates = visits.duplicated(keep=False)
num_full_duplicates = full_duplicates.sum()

print(f"Number of full row duplicates: {num_full_duplicates}")

# Filter rows with duplicate keys but different content
key_only_duplicates = duplicate_key_mask & ~full_duplicates
key_only_df = visits[key_only_duplicates].sort_values(by='visit_id')


print(f"Number of duplicate keys with different row content: {key_only_df.shape[0]}")
display(key_only_df)

Number of full row duplicates: 0
Number of duplicate keys with different row content: 0


,visit_id,patient_id,clinic_id,visit_date,procedure_code,amount,quantity,source,name,email,birth_date,region_code,procedure_name,specialty,standard_cost,region_name


In [24]:
# Select numeric columns
numeric_cols = visits.select_dtypes(include=['number']).columns

numeric_summary = visits[numeric_cols].agg(['min', 'max', 'mean', 'std']).T
numeric_summary.columns = ['Min', 'Max', 'Mean', 'Std Dev']

display(numeric_summary)

,Min,Max,Mean,Std Dev
amount,-479.50,1737.687050,75.859027,170.690565
quantity,-328.05,1223.405564,76.377483,139.840720
standard_cost,1.00,1164.970000,550.942448,370.647101


In [25]:
# Identify datetime columns
date_cols = visits.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns

# Compute min and max for each datetime column
date_ranges = visits[date_cols].agg(['min', 'max']).T
date_ranges.columns = ['Earliest', 'Latest']

# Display the result
print("Date ranges for all datetime columns:")
display(date_ranges)

Date ranges for all datetime columns:


,Earliest,Latest
visit_date,2024-01-01,2024-06-30
birth_date,1934-11-25,2006-08-12


**Validation Notes:**

- Missing values:
    - `clinic_id`: 62 missing values.
    - `amount`: 34 missing values.
    - `birth_date`: 8 missing values (coerced from invalid formats).
    - `region_code` and `region_name`: 69 missing values. These are likely due to missing region codes in the original patients data that could not be joined with the region lookup.
    - `standard_cost`: 10 missing values. This is likely due to the one missing value in the original `procedure_catalog` and potentially missing procedure codes in the visits data that didn't match the catalog (though the earlier check showed 0 missing procedures).

- Duplicates:
    - No full row duplicates found.
    - 2 duplicate `visit_id` entries were found, but with different content (specifically, one has a missing `amount`).

- Outliers:
    - Negative amounts exist in both Q1 and Q2 visits (10 in Q1, 4 in Q2). These might represent refunds or errors and need further investigation depending on the analysis goals.
    - Quantities are all positive and within a reasonable range (1-3).
    - Birth dates range from 1934 to 2006, which seems plausible for a healthcare clinic dataset.
    - Standard costs range from 1 to 1164.97, which also seems reasonable for medical procedures.


**These checks ensure that the processed dataset is internally consistent
and suitable for downstream analysis.**



---



## Output Dataset

In [26]:
# Display the first 10 rows of the final prepared dataset
visits.head(10)

,visit_id,patient_id,clinic_id,visit_date,procedure_code,amount,quantity,source,name,email,birth_date,region_code,procedure_name,specialty,standard_cost,region_name
0,VQ1-10000,PT1026,CL103,2024-01-17,PROC013,96.02,1.0,Q1,Emerson Martin,emerson.martin@health.org,1981-08-14,SW,Imaging 23,Radiology,822.83,Southwest
1,VQ1-10001,PT1087,CL104,2024-01-16,PROC026,-479.50,1.0,Q1,Sawyer Miller,sawyer.miller@mail.com,1970-11-23,NE,Procedure 42,Neurology,421.40,Northeast
2,VQ1-10002,PT1015,CL109,2024-01-06,PROC023,61.93,2.0,Q1,Jordan Davis,jordan.davis@health.org,1993-02-26,NaN,Lab Panel 30,Urology,671.73,NaN
3,VQ1-10003,PT1010,CL101,2024-03-10,PROC014,32.60,1.0,Q1,Parker Williams,parker.williams@example.com,1948-03-12,NaN,Therapy 25,Oncology,228.30,NaN
4,VQ1-10004,PT1087,CL108,2024-03-06,PROC017,187.82,1.0,Q1,Sawyer Miller,sawyer.miller@mail.com,1970-11-23,NE,Screening 42,Dermatology,995.99,Northeast
5,VQ1-10005,PT1034,CL106,2024-02-03,PROC023,141.28,1.0,Q1,Taylor Hernandez,taylor.hernandezexample.com,1941-03-03,SE,Lab Panel 30,Urology,671.73,Southeast
6,VQ1-10006,PT1078,CL103,2024-01-09,PROC021,216.12,1.0,Q1,Casey Martin,casey.martin@health.org,1969-03-26,NaN,Imaging 39,Urology,148.15,NaN
7,VQ1-10007,PT1079,CL104,2024-01-21,PROC018,220.24,2.0,Q1,Dakota Taylor,dakota.taylor@mail.com,2000-03-20,MW,Follow-up 43,Pediatrics,196.12,Midwest
8,VQ1-10008,PT1066,CL108,2024-03-14,PROC020,238.44,3.0,Q1,Alex Brown,alex.brown@mail.com,1989-01-27,SW,Lab Panel 44,Cardiology,1160.04,Southwest
9,VQ1-10009,PT1113,CL106,2024-01-28,PROC020,193.58,2.0,Q1,Finley Johnson,finley.johnson@health.org,1996-05-23,SE,Lab Panel 44,Cardiology,1160.04,Southeast


**Data Prep Log**

- Issue: Inconsistent column naming conventions across `visits_q1`, `visits_q2`, `patients`, `region_lookup`, and `procedure_catalog`. Some column names used PascalCase or had spaces.
- Action: Standardized all column names to lowercase with underscores using the `standardize_column_names` function.

- Issue: Data type inconsistencies, particularly with dates in `visits_q2` and `patients`, amount in `visits_q2`, and standard cost in `procedure_catalog`.
- Action: Converted 'visit_date' in `visits_q1` and `visits_q2` to datetime objects. Converted 'birth_date' in `patients` to datetime objects, coercing invalid values to NaT. Converted 'amount' in `visits_q2` and 'standard_cost' in `procedure_catalog` to numeric types, coercing errors.

- Issue: Duplicate rows found in `visits_q1`.
- Action: Removed the duplicate row from `visits_q1`.

- Issue: A 'TOTAL' row existed in `visits_q2`.
- Action: Removed the 'TOTAL' row from `visits_q2`.

- Issue: Inconsistent casing in `region_code` in the `patients` DataFrame ('SW' vs 'Sw').
- Action: Standardized the `region_code` column in the `patients` DataFrame to uppercase.

- Issue: Duplicate region codes in `region_lookup`.
- Action: Removed duplicate region codes from `region_lookup`, keeping the first occurrence.

- Issue: Data was spread across multiple files (`patients.csv`, `visits_q1.csv`, `visits_q2.csv`, `region_lookup copy.xlsx`, `procedure_catalog.json`).
- Action: Vertically concatenated `visits_q1` and `visits_q2`. Performed left joins to merge `visits` with `patients`, `procedure_catalog`, and `region_lookup` based on relevant key columns (`patient_id`, `procedure_code`, `region_code`).

- Remaining limitations:
    - Significant number of missing values in `clinic_id`, `amount`, `birth_date`, `region_code`, `region_name`, and `standard_cost` in the final merged dataset. These missing values were retained to avoid dropping potentially valuable visit records but would need further imputation or handling depending on the analysis.
    - Negative amounts are present in the visit data. These were not removed and would require further investigation to determine if they represent refunds or data entry errors.
    - Two duplicate `visit_id` entries exist with different content, suggesting a potential data issue that might need further investigation or a decision on which record to keep.

In [27]:
# Transform visits into a csv file named integrated_dataset.csv
visits.to_csv(OUT_DIR / "integrated_dataset.csv", index=False)

**The resulting dataset represents the single source of truth for downstream EDA and analysis.**